In [1]:
#Importing libraries
import numpy as np
import scipy as sp
from scipy.optimize import minimize

In [22]:
#Constructing the function
def f(p):
    #a0=1.04
    a0=np.random.normal(1.04,0.29)
    #r=0.96
    r=np.random.normal(0.96,0.18)
    pcosdelta=1/a0+0.5*r*(p**2)
    return pcosdelta
    #den=pcosdelta-1j*p
    #return den

In [40]:
#Getting data points
p=np.linspace(1,10,10)
fp=[]
for i in p:
    fp.append(f(i))

In [4]:
#Defining least fit norms

def l2(p,fv,g):
    l2=0
    for i in range(len(fv)):
        l2=l2+(fv[i]-g(p[i]))**2
    return l2

def l1(p,fv,g):
    l1=0
    for i in range(len(fv)):
        l1=l1+(fv[i]-g(p[i]))
    return l1    

def linf(p,fv,g):
    linf=0
    err=[]
    for i in range(len(fv)):
        err.append(fv[i]-g(p[i]))
    i=np.argmax(err)
    linf=err[i]
    return linf

In [5]:
# Define the L2 norm function for optimization for constant function
def l2(c, p, fv):
    cr, ci = c[0], c[1]
    sum = 0
    for i in range(len(fv)):
        sum=sum+np.abs(fv[i] - (cr + ci * 1j))**2
    return sum
    
a0 = [0, 0]

# Minimize the L2 norm function
res = minimize(l2, a0, args=(p, fp))

coeff=[]
for i in range(2):
    coeff.append(res.x[i])

kaisq=l2(a0,p,fp)
kaisq

6223.323658215648

In [52]:
#Constructing Pade Approximant using polynomial fit 
def f(p):
    #a0=1.04
    a0=np.random.normal(1.04,0.29)
    #r=0.96
    r=np.random.normal(0.96,0.18)
    pcosdelta=1/a0+0.5*r*(p**2)
    den=pcosdelta-1j*p
    return den

#1. Getting data points
p=np.linspace(1,10,10)
fp=[]
for i in p:
    fp.append(f(i))
# 2. Finding the best numerator least fit
coeff=0
kaisqo=10e6
kaisq=10e5
num=5
for i in range(5):
    coeffo=coeff
    coeff=np.polyfit(p,fp,i)
    poly=np.poly1d(coeff)
    kaisqo=kaisq
    kaisq=0
    for j in range(num):
        kaisq=kaisq+(fp[j]-poly(p[j]))**2   
    if (kaisqo<=kaisq):
        coeff=coeffo
        break
print(coeff)
poly=np.poly1d(coeff)

[ 0.28680248  1.48397759 -0.63847027]


array([-5.57362561,  0.39941096])

In [74]:
#Trying averaging of zeroes
r1,r2=[],[]
for i in range (10):
    #1.Obtaining the roots 
    p=np.linspace(1,10,10)
    fp=[]
    for i in p:
        fp.append(f(i))
    coeff=0
    kaisqo=10e6
    kaisq=10e5
    num=5
    for i in range(5):
        coeffo=coeff
        coeff=np.polyfit(p,fp,i)
        poly=np.poly1d(coeff)
        kaisqo=kaisq
        kaisq=0
        for j in range(num):
            kaisq=kaisq+(fp[j]-poly(p[j]))**2   
        if (kaisqo<=kaisq):
            coeff=coeffo
            break
    poly=np.poly1d(coeff)
    roots=np.roots(poly)
    r1.append(roots[0])
    for i in range(1,len(roots)):
        if (roots[i].imag==-roots[i-1].imag):
            r2.append(roots[i])
        else:
            r1.append(roots[i])
avgr1=sum(r1)/len(r1)
std1=np.std(r1)
print(avgr1,std1)


avgr2=sum(r2)/len(r2)
std2=np.std(r2)
print(avgr2,std2)

(6.210626630083831+1.6239764708839055j) 26.78576069742969
(1.8412353352066004-2.657416043264573j) 3.242359462334183


In [1]:
import numpy as np
from scipy.optimize import minimize

# Constructing Pade Approximant using polynomial fit
def f(p):
    a0 = np.random.normal(1.04, 0.29)
    r = np.random.normal(0.96, 0.18)
    pcosdelta = 1 / a0 + 0.5 * r * (p**2)
    den = pcosdelta - 1j * p
    return den

# 1. Getting data points
p = np.linspace(1, 10, 10)
fp = np.array([f(i) for i in p])

# 2. Define the error function for L1 norm
def l1_error(coeff, p, fp, m, n):
    num_coeff_real = coeff[:m + 1]
    num_coeff_imag = coeff[m + 1:2 * (m + 1)]
    den_coeff_real = [1.0] + coeff[2 * (m + 1):2 * (m + 1) + n]  # Fix constant term to 1
    den_coeff_imag = [0.0] + coeff[2 * (m + 1) + n:]  # Fix imaginary part of constant term to 0

    def pade_approx(x):
        num = np.poly1d(num_coeff_real + 1j * num_coeff_imag)(x)
        den = np.poly1d(den_coeff_real + 1j * den_coeff_imag)(x)
        return num / den

    error = 0
    for i in range(len(p)):
        error += np.abs(fp[i] - pade_approx(p[i]))
    return error

# Initial guess for coefficients (real and imaginary parts)
m = 3  # Degree of numerator
n = 3  # Degree of denominator
initial_guess = np.random.randn(2 * (m + 1 + n))

# Optimize the coefficients using L1 norm
res = minimize(l1_error, initial_guess, args=(p, fp, m, n), method='Nelder-Mead')

# Extract optimized coefficients
opt_coeff = res.x
num_coeff_real = opt_coeff[:m + 1]
num_coeff_imag = opt_coeff[m + 1:2 * (m + 1)]
den_coeff_real = [1.0] + opt_coeff[2 * (m + 1):2 * (m + 1) + n]
den_coeff_imag = [0.0] + opt_coeff[2 * (m + 1) + n:]

# Normalize the coefficients to ensure the constant term of the denominator is 1
denominator_poly = np.poly1d(den_coeff_real + 1j * den_coeff_imag)
denominator_const_term = denominator_poly[0]  # The constant term of the denominator

numerator_poly = np.poly1d(num_coeff_real + 1j * num_coeff_imag) / denominator_const_term
denominator_poly = denominator_poly / denominator_const_term

# Displaying the Pade approximant
print("Pade Approximant:")
print("Numerator:", numerator_poly)
print("Denominator:", denominator_poly)

# Evaluating the Pade approximant at points p
pade_approx_values = numerator_poly(p) / denominator_poly(p)
print("Pade Approximant values:", pade_approx_values)
print("Original function values:", fp)
numerator_poly.r

Pade Approximant:
Numerator:                        3                     2
(0.05593 + -0.04785j) x + (14.67 + 0.6357j) x + (-0.2052 + 0.07437j) x + (0.9243 + -0.3198j)
Denominator:                      2
(0.07103 + 0.1054j) x + (3.389 + 0.5291j) x + 1
Pade Approximant values: [ 3.40427777 -0.40756521j  7.19881034 -1.07476133j
 10.90997136 -1.97667333j 14.44951422 -3.05047658j
 17.80101006 -4.26196372j 20.9636878  -5.58551419j
 23.94221777 -7.00008444j 26.74365544 -8.48788179j
 29.37619844-10.033725j   31.84856336-11.62463643j]
Original function values: [ 1.07854652 -1.j  3.05829027 -2.j  5.49447173 -3.j  7.265722   -4.j
 14.57503005 -5.j 21.06174821 -6.j 22.33061703 -7.j 26.49371585 -8.j
 34.77087062 -9.j 49.30774844-10.j]


array([-1.45810544e+02-136.10161289j, -4.12759815e-02  -0.2564417j ,
        5.51656986e-02  +0.25047503j])

In [2]:
import numpy as np
from scipy.optimize import minimize

# Constructing Pade Approximant using polynomial fit
def f(p):
    a0 = np.random.normal(1.04, 0.29)
    r = np.random.normal(0.96, 0.18)
    pcosdelta = 1 / a0 + 0.5 * r * (p**2)
    den = pcosdelta - 1j * p
    return den

# 1. Getting data points
p = np.linspace(1, 10, 10)
fp = np.array([f(i) for i in p])

# 2. Define the error function for L2 norm
def l2_error(coeff, p, fp, m, n):
    num_coeff_real = coeff[:m + 1]
    num_coeff_imag = coeff[m + 1:2 * (m + 1)]
    den_coeff_real = [1.0] + coeff[2 * (m + 1):2 * (m + 1) + n]  # Fix constant term to 1
    den_coeff_imag = [0.0] + coeff[2 * (m + 1) + n:]  # Fix imaginary part of constant term to 0

    def pade_approx(x):
        num = np.poly1d(num_coeff_real + 1j * num_coeff_imag)(x)
        den = np.poly1d(den_coeff_real + 1j * den_coeff_imag)(x)
        return num / den

    error = 0
    for i in range(len(p)):
        error += np.abs(fp[i] - pade_approx(p[i]))**2
    return error

# Initial guess for coefficients (real and imaginary parts)
m = 3  # Degree of numerator
n = 3  # Degree of denominator
initial_guess = np.random.randn(2 * (m + 1 + n))

# Optimize the coefficients using L2 norm
res = minimize(l2_error, initial_guess, args=(p, fp, m, n), method='Nelder-Mead')

# Extract optimized coefficients
opt_coeff = res.x
num_coeff_real = opt_coeff[:m + 1]
num_coeff_imag = opt_coeff[m + 1:2 * (m + 1)]
den_coeff_real = [1.0] + opt_coeff[2 * (m + 1):2 * (m + 1) + n]
den_coeff_imag = [0.0] + opt_coeff[2 * (m + 1) + n:]

# Normalize the coefficients to ensure the constant term of the denominator is 1
denominator_poly = np.poly1d(den_coeff_real + 1j * den_coeff_imag)
denominator_const_term = denominator_poly[0]  # The constant term of the denominator

numerator_poly = np.poly1d(num_coeff_real + 1j * num_coeff_imag) / denominator_const_term
denominator_poly = denominator_poly / denominator_const_term

# Displaying the Pade approximant
print("Pade Approximant:")
print("Numerator:", numerator_poly)
print("Denominator:", denominator_poly)

# Evaluating the Pade approximant at points p
pade_approx_values = numerator_poly(p) / denominator_poly(p)
print("Pade Approximant values:", pade_approx_values)
print("Original function values:", fp)
numerator_poly.r

Pade Approximant:
Numerator:                       3                      2
(-0.1696 + -0.3738j) x + (0.5737 + 0.7689j) x + (0.3014 + -0.4218j) x + (0.5522 + -1.309j)
Denominator:                         2
(-0.01834 + -0.04949j) x + (-0.01555 + -0.165j) x + 1
Pade Approximant values: [ 1.53323475-1.04213962j  2.74452208-0.69055109j  4.65782771-1.73394164j
  8.43294464-3.68509375j 13.77495687-5.42355347j 19.93645117-6.68586482j
 26.49812875-7.59688902j 33.2813269 -8.29597041j 40.20857948-8.87335681j
 47.24159844-9.38173559j]
Original function values: [ 1.65276589 -1.j  3.00132717 -2.j  6.52500642 -3.j  6.94675596 -4.j
 11.45337389 -5.j 14.70129133 -6.j 37.39567638 -7.j 29.99719471 -8.j
 37.72659277 -9.j 48.48513694-10.j]


array([ 2.41171246-1.38192259j,  0.72568031+1.17390297j,
       -0.85405108-0.29093011j])

In [3]:
import numpy as np
from scipy.optimize import minimize

# Constructing Pade Approximant using polynomial fit
def f(p):
    a0 = np.random.normal(1.04, 0.29)
    r = np.random.normal(0.96, 0.18)
    pcosdelta = 1 / a0 + 0.5 * r * (p**2)
    den = pcosdelta - 1j * p
    return den

# 1. Getting data points
p = np.linspace(1, 10, 10)
fp = np.array([f(i) for i in p])

# 2. Define the error function for Minimax (Chebyshev) norm
def minimax_error(coeff, p, fp, m, n):
    num_coeff_real = coeff[:m + 1]
    num_coeff_imag = coeff[m + 1:2 * (m + 1)]
    den_coeff_real = [1.0] + coeff[2 * (m + 1):2 * (m + 1) + n]
    den_coeff_imag = [0.0] + coeff[2 * (m + 1) + n:]

    def pade_approx(x):
        num = np.poly1d(num_coeff_real + 1j * num_coeff_imag)(x)
        den = np.poly1d(den_coeff_real + 1j * den_coeff_imag)(x)
        return num / den

    error = np.max([np.abs(fp[i] - pade_approx(p[i])) for i in range(len(p))])
    return error

# Initial guess for coefficients (real and imaginary parts)
m = 3  # Degree of numerator
n = 3  # Degree of denominator
initial_guess = np.random.randn(2 * (m + 1 + n))

# Optimize the coefficients using Minimax norm
res = minimize(minimax_error, initial_guess, args=(p, fp, m, n), method='Nelder-Mead')

# Extract optimized coefficients
opt_coeff = res.x
num_coeff_real = opt_coeff[:m + 1]
num_coeff_imag = opt_coeff[m + 1:2 * (m + 1)]
den_coeff_real = [1.0] + opt_coeff[2 * (m + 1):2 * (m + 1) + n]
den_coeff_imag = [0.0] + opt_coeff[2 * (m + 1) + n:]

# Normalize the coefficients to ensure the constant term of the denominator is 1
denominator_poly = np.poly1d(den_coeff_real + 1j * den_coeff_imag)
denominator_const_term = denominator_poly[0]  # The constant term of the denominator

numerator_poly = np.poly1d(num_coeff_real + 1j * num_coeff_imag) / denominator_const_term
denominator_poly = denominator_poly / denominator_const_term

# Displaying the Pade approximant
print("Pade Approximant (Minimax Norm):")
print("Numerator:", numerator_poly)
print("Denominator:", denominator_poly)

# Evaluating the Pade approximant at points p
pade_approx_values = numerator_poly(p) / denominator_poly(p)
print("Pade Approximant values:", pade_approx_values)
print("Original function values:", fp)
numerator_poly.r

Pade Approximant (Minimax Norm):
Numerator:                    3                      2
(3.098 + 0.1257j) x + (-0.9609 + 1.977j) x + (0.1639 + 0.2563j) x + (-2.445 + 1.685j)
Denominator:                     2
(0.9389 + 0.4297j) x + (-0.8727 + -0.348j) x + 1
Pade Approximant values: [ 0.1544537  +3.78149559j  6.73056354 +1.40504191j
 10.35399731 -0.81758655j 13.37282158 -2.46794874j
 16.2438501  -3.88441736j 19.06520239 -5.19341021j
 21.86603116 -6.44588997j 24.65713443 -7.66544986j
 27.44316445 -8.86430632j 30.22635605-10.04935641j]
Original function values: [ 1.32129144 -1.j  2.95065464 -2.j  3.86923391 -3.j  8.96231932 -4.j
 15.41815974 -5.j 15.01612048 -6.j 20.05964921 -7.j 19.64595175 -8.j
 19.19562805 -9.j 38.47502899-10.j]


array([ 0.99877899-0.51059352j, -0.47175378-0.92479233j,
       -0.24325187+0.78566288j])